# H3000 - Neblina

Hello :) Welcome to the rocketPy script for Neblina, Antares' project for the 2026 LASC.

## Nominal rocket
First, we'll define the idealized version of Neblina, without including any measurement uncertainties (which will be taken into account further down, in the 'Stochastic rocket' section)

### Enviroment
Here, we define the launch place and weather conditions.

In [ ]:
from rocketpy import Environment, SolidMotor, Rocket, Flight, MonteCarlo
# import datetime
from datetime import datetime, date, timedelta
import math
import copy
pi = math.pi
# We import these lines for debugging purposes, only works on Jupyter Notebook
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [ ]:
env = Environment(latitude= -21.90795, longitude= -48.96156, elevation= 495,
                  timezone= 'America/Sao_Paulo')

# Launch day
launchDay = datetime(2026, 9, 3)
env.set_date(
    (launchDay.year, launchDay.month, launchDay.day, 13, 30, 0)
    )  # Hour given in UTC time

env.set_atmospheric_model(type="custom_atmosphere",
                          temperature= 298,
                          wind_u=[
                              (0, 3.6), # at 0 m
                              (1000, 10) # at 1000 m
                          ])

# env.set_atmospheric_model(type="Windy", file="GFS")


In [ ]:
env.info()

### Motor
Here we define Yaripo's physical and geometric properties

In [ ]:
Yaripo = SolidMotor(
    thrust_source="../neblinaPy/data/Yaripo_teste_est_2.eng",
    coordinate_system_orientation= "combustion_chamber_to_nozzle",
    dry_mass= 0.976 + 3.432 + 3.017 + 0.991,   # TP + comb. chamb. + nozzle + therm. ins.     updated 28/07/26
    center_of_dry_mass_position= 0.358,   # Taken from CREO mass properties     updated 28/07/26
    dry_inertia= (0.54060753, 0.54060196, 0.025867792),   # Taken from CREO mass properties     updated 28/07/26
    nozzle_radius= 60 / 2000,
    grain_number=4,
    grain_density=1815,
    grain_outer_radius= 116 /2000,
    grain_initial_inner_radius= 40 / 2000,
    grain_initial_height= 129 / 1000,
    grain_separation= 2 /1000,
    grains_center_of_mass_position = 261 /1000,
    nozzle_position= 723.3 /1000,
    throat_radius= 20.6 / 2000,
)

In [ ]:
# Check if the thrust data doesn't have any NaNs of Infs
import numpy as np

print("Propellant Mass:", Yaripo.propellant_initial_mass)
print("Total Impulse:", Yaripo.total_impulse)

thrust_data = Yaripo.thrust.source  # NumPy array of [time, thrust]
print("Contains NaNs:", np.isnan(thrust_data).any())
print("Contains Infs:", np.isinf(thrust_data).any())

with open("../neblinaPy/data/Yaripo_teste_est_2.eng", "rb") as f:
    print(f.readline())

print(thrust_data)

In [ ]:
Yaripo.info()
Yaripo.draw()

### Rocket body
Here we define the shape and postions of our rocket

In [ ]:
neblina = Rocket(
    radius= 156 /2000,
    mass= 17.186,   # mass without motor
    inertia= (5.38, 5.38, 0.075405296),   # Taken from Creo's Mass Properties function   #TODO: update later
    power_off_drag="../neblinaPy/data/cd_poweroff.CSV",   #TODO: update when CFD is done
    power_on_drag="../neblinaPy/data/cd_poweron.CSV",   #TODO: update when CFD is done
    center_of_mass_without_motor= 1.282,
    coordinate_system_orientation="nose_to_tail",
)

neblina.add_motor(Yaripo, position= 2.1755)

rail_buttons = neblina.set_rail_buttons(
    upper_button_position= 1.500,
    lower_button_position= 2.650,
    angular_position=45,
)

nose_cone = neblina.add_nose(
    length= 550 /1000, kind = "von karman", position = 0
)

fin_set = neblina.add_trapezoidal_fins(
    n=4,
    root_chord=0.300,
    tip_chord=0.075,
    span=0.170,
    position= 2.495,
    sweep_angle= 50,
    # airfoil=("../data/airfoils/NACA0012-radians.txt","radians"), #TODO: this
)

tail = neblina.add_tail(
    top_radius= 156 /2000, bottom_radius= 100 /2000, length= 198 /1000, position= 2.694
)

In [ ]:
# Here we create copies of Neblina, se we can add different parachute configs to each one
# to have a flight with no parachute ejection, one with the main opening on apogee, and the nominal case
neblina_ballistic = copy.deepcopy(neblina)
neblina_main_on_apogee = copy.deepcopy(neblina)

if neblina.parachutes != [] or neblina_ballistic.parachutes != [] or neblina_main_on_apogee.parachutes != []:
    print("WARNING: Some of the rockets already have parachutes defined. Remove them and add again.")
    print(
        neblina.parachutes, neblina_ballistic.parachutes, neblina_main_on_apogee.parachutes
    )

In [ ]:
# Defining the geometric and aerodynamic properties of the parachutes

# "Main" parachute
cd_main = 0.87
mainDiam = 3.2
spillHoleArea = pi * ((3.2*0.2) /2)**2
mainRadius = mainDiam / 2
mainArea = (pi * (mainDiam / 2) ** 2) - spillHoleArea

# Reefed parachute
cd_drogue = 0.22
drogueDiam = 1.167
drogueRadius = drogueDiam / 2
drogueArea = (pi * drogueRadius ** 2) - spillHoleArea

# Add parachutes to each version of the rocket

# Nominal:
main = neblina.add_parachute(
    name="main",
    cd_s=cd_main * mainArea,
    trigger= 500,      # AGL ejection altitude
    sampling_rate=100,
    lag=1,
    noise=(0, 8.3, 0.5),
    radius= mainRadius,
    height=0.7 * mainRadius,
    porosity=0.0432,
)
drogue = neblina.add_parachute(
    name="drogue",
    cd_s= cd_drogue * drogueArea,
    trigger="apogee",  # ejection at apogee
    sampling_rate=100,
    lag=2,
    noise=(0, 8.3, 0.5),
    radius= drogueRadius,
    porosity=0.0432,
)

# Main on apogee:
mainOnApogee = neblina_main_on_apogee.add_parachute(
    name="main(on apogee)",
    cd_s=cd_main * mainArea,
    trigger= "apogee",
    sampling_rate=100,
    lag=2,
    noise=(0, 8.3, 0.5),
    radius= mainRadius,
    height=0.7 * mainRadius,
    porosity=0.0432,
)


In [ ]:
print( "Nominal case parachutes: ", neblina.parachutes, "\n", "Main on apogee case parachutes", neblina_main_on_apogee.parachutes
    )

In [ ]:
# Remove all parachutes if necessary

neblina.parachutes.clear()
neblina_main_on_apogee.parachutes.clear()
print( "Nominal case parachutes: ", neblina.parachutes, "\n", "Main on apogee case parachutes", neblina_main_on_apogee.parachutes
    )

In [ ]:
neblina.plots.static_margin()
neblina.plots.stability_margin()

In [ ]:
neblina.draw()
neblina.draw(filename="neblina.png")

### Run simulation
Run it !

In [ ]:
railLength = 6.0   # meters
railIncl = 80   # degrees
heading = 90   # degrees

flight = Flight(
    rocket=neblina, environment=env, rail_length=railLength, inclination=railIncl, heading=heading, name="Nominal"
    )
flight_ballistic = Flight(
    rocket=neblina_ballistic, environment=env, rail_length=railLength, inclination=railIncl, heading=heading, name= "Ballistic"
    )
flight_MoA = Flight(
    rocket=neblina_main_on_apogee, environment=env, rail_length=railLength, inclination=railIncl, heading=heading, name= "Main on apogee"
    )

In [ ]:
# Print mass and force balance to check if the simulation won't just give up before lift-off
rocket_mass_kg = flight.rocket.total_mass(0)
rocket_weight_N = rocket_mass_kg * 9.80665

print(f"Total Rocket Mass:   {rocket_mass_kg:.2f} kg")
print(f"Total Rocket Weight: {rocket_weight_N:.2f} N")
print(f"Motor Max Thrust:    {Yaripo.max_thrust:.2f} N")
print(f"Thrust at t=1.0s:    {Yaripo.thrust(1.0):.2f} N")
print(f"Out of Rail Time:    {flight.out_of_rail_time}")

In [ ]:
flight.info()

In [ ]:
from rocketpy.plots.compare import CompareFlights
comparison = CompareFlights([flight, flight_ballistic, flight_MoA])
comparison.trajectories_3d(legend=True, filename="3d_comparison.png")
comparison.trajectories_2d(plane="xy", legend=True, filename="xy_comparison.png")

# flight.plots.trajectory_3d()
# flight2.plots.trajectory_3d()

In [ ]:
from rocketpy.simulation import FlightDataExporter
FlightDataExporter(flight).export_kml(
    file_name="fullNominal.kml",
    extrude=True,
    altitude_mode="relativetoground",
)

FlightDataExporter(flight_ballistic).export_kml(
    file_name="ballistic.kml",
    extrude=True,
    altitude_mode="relativetoground",
)

FlightDataExporter(flight_MoA).export_kml(
    file_name="mainOnApogee.kml",
    extrude=True,
    altitude_mode="relativetoground",
)

### Further analysis

In [ ]:
from rocketpy.utilities import apogee_by_mass

apogee_by_mass(
    flight=flight, min_mass=5, max_mass=20, points=10, plot=True
    )

# Aerodynamics plots
flight.plots.fluid_mechanics_data()

# Velocity and acceleration plots
flight.plots.linear_kinematics_data()